# Сбор данных

## Анализ застройки

Код для выгрузки данных по зданиям с платформы Overpass Turbo

```
[out:json][timeout:60];
(
  way["building"](51.49, 104.10, 51.54, 104.23);
  relation["building"](51.49, 104.10, 51.54, 104.23);
);
out body;
>;
out skel qt;
```

### Удаление лишних столбцов

In [ ]:
!pip install geopandas

In [ ]:
import geopandas as gpd
import pandas as pd
import requests
import time

In [ ]:
gdf = gpd.read_file("export.geojson")
gdf.head()

,id,@id,addr:city,addr:housenumber,addr:place,addr:postcode,addr:street,amenity,brand,brand:en,...,phone,ref,roof:colour,roof:shape,shop,source:addr,start_date,tourism,website,geometry
0,way/233398440,way/233398440,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,"POLYGON ((104.22613 51.50023, 104.22651 51.500..."
1,way/295659409,way/295659409,None,169,микрорайон Гагарина,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,"POLYGON ((104.15024 51.52243, 104.1501 51.5224..."
2,way/295659410,way/295659410,None,192,микрорайон Гагарина,None,None,None,None,None,...,None,None,silver,gabled,None,None,None,None,None,"POLYGON ((104.15338 51.51682, 104.15244 51.516..."
3,way/295659411,way/295659411,None,197,микрорайон Гагарина,None,None,None,None,None,...,None,None,silver,gabled,None,None,None,None,None,"POLYGON ((104.15638 51.51899, 104.1552 51.5189..."
4,way/295659413,way/295659413,None,203,микрорайон Гагарина,None,None,None,None,None,...,None,None,silver,gabled,None,None,None,None,None,"POLYGON ((104.15733 51.51971, 104.15614 51.519..."


In [ ]:
print("Столбцы:", gdf.columns)
print("Количество непустых значений в каждом столбце: ", gdf.notnull().sum())

Столбцы: Index(['id', '@id', 'addr:city', 'addr:housenumber', 'addr:place',
       'addr:postcode', 'addr:street', 'amenity', 'brand', 'brand:en',
       'brand:ru', 'brand:wikidata', 'building', 'building:levels',
       'contact:phone', 'contact:website', 'description', 'email',
       'fuel:diesel', 'fuel:discount', 'fuel:octane_100', 'fuel:octane_92',
       'fuel:octane_95', 'internet_access', 'leisure', 'name', 'name:en',
       'name:ja', 'name:ru', 'opening_hours', 'operator', 'operator:en',
       'operator:wikidata', 'phone', 'ref', 'roof:colour', 'roof:shape',
       'shop', 'source:addr', 'start_date', 'tourism', 'website', 'geometry'],
      dtype='object')
Количество непустых значений в каждом столбце:  id                   946
@id                  946
addr:city             44
addr:housenumber     480
addr:place           155
addr:postcode         18
addr:street          367
amenity                6
brand                  1
brand:en               1
brand:ru               

In [ ]:
cols_to_remove = [
    'addr:city', 'amenity', 'brand', 'brand:en', 'brand:ru', 'brand:wikidata',
    'contact:phone', 'contact:website', 'description', 'email',
    'fuel:diesel', 'fuel:discount', 'fuel:octane_100', 'fuel:octane_92', 'fuel:octane_95',
    'internet_access', 'leisure', 'opening_hours', 'operator', 'operator:en',
    'operator:wikidata', 'phone', 'ref', 'shop', 'source:addr', 'start_date',
    'tourism', 'website', 'addr:postcode', 'name:en', 'name:ja', 'name:ru',
    'roof:colour', 'roof:shape', 'name'
]

In [ ]:
cols_exist = [col for col in cols_to_remove if col in gdf.columns]

gdf_clean = gdf.drop(columns=cols_exist)

print("Удалены столбцы:", cols_exist)
print("Остались столбцы:", gdf_clean.columns)

Удалены столбцы: ['addr:city', 'amenity', 'brand', 'brand:en', 'brand:ru', 'brand:wikidata', 'contact:phone', 'contact:website', 'description', 'email', 'fuel:diesel', 'fuel:discount', 'fuel:octane_100', 'fuel:octane_92', 'fuel:octane_95', 'internet_access', 'leisure', 'opening_hours', 'operator', 'operator:en', 'operator:wikidata', 'phone', 'ref', 'shop', 'source:addr', 'start_date', 'tourism', 'website', 'addr:postcode', 'name:en', 'name:ja', 'name:ru', 'roof:colour', 'roof:shape', 'name']
Остались столбцы: Index(['id', '@id', 'addr:housenumber', 'addr:place', 'addr:street',
       'building', 'building:levels', 'geometry'],
      dtype='object')


In [ ]:
gdf_clean = gdf_clean.rename(columns={
    'building:levels': 'floors',
    'addr:housenumber': 'house_number',
    'addr:street': 'street',
    'addr:place': 'district'
})

In [ ]:
gdf_clean.to_file("baikalsk_clean.geojson", driver="GeoJSON")

### Дополнение

In [ ]:
gdf = gpd.read_file("baikalsk_clean.geojson")
gdf.head()

,id,@id,house_number,district,street,building,floors,geometry
0,way/233398440,way/233398440,None,None,None,train_station,None,"POLYGON ((104.22613 51.50023, 104.22651 51.500..."
1,way/295659409,way/295659409,169,микрорайон Гагарина,None,yes,3,"POLYGON ((104.15024 51.52243, 104.1501 51.5224..."
2,way/295659410,way/295659410,192,микрорайон Гагарина,None,yes,3,"POLYGON ((104.15338 51.51682, 104.15244 51.516..."
3,way/295659411,way/295659411,197,микрорайон Гагарина,None,yes,3,"POLYGON ((104.15638 51.51899, 104.1552 51.5189..."
4,way/295659413,way/295659413,203,микрорайон Гагарина,None,yes,3,"POLYGON ((104.15733 51.51971, 104.15614 51.519..."


In [ ]:
non_null_counts = gdf.notnull().sum()

print("Количество непустых значений в каждом столбце:")
print(non_null_counts)

Количество непустых значений в каждом столбце:
id              946
@id             946
house_number    480
district        155
street          367
building        946
floors           90
geometry        946
dtype: int64


In [ ]:
# Смотрим уникальные значения в столбце 'building'
unique_buildings = gdf['building'].dropna().unique()
print("Уникальные значения в столбце 'building':")
for b in sorted(unique_buildings):
    print(b)

Уникальные значения в столбце 'building':
apartments
commercial
garages
hospital
house
industrial
kiosk
public
residential
retail
roof
train_station
yes


In [ ]:
gdf['floors'] = pd.to_numeric(gdf['floors'], errors='coerce')

most_common_floors = {}
for b in gdf['building'].dropna().unique():
    mode = gdf.loc[gdf['building'] == b, 'floors'].dropna().mode()
    most_common_floors[b] = mode.iloc[0] if not mode.empty else 1

gdf['floors'] = gdf.apply(
    lambda row: row['floors'] if pd.notnull(row['floors'])
    else most_common_floors.get(row['building'], 1),
    axis=1
)

print(gdf[['building', 'floors']].head(20))

         building  floors
0   train_station     1.0
1             yes     3.0
2             yes     3.0
3             yes     3.0
4             yes     3.0
5             yes     3.0
6             yes     3.0
7             yes     3.0
8             yes     3.0
9             yes     3.0
10            yes     3.0
11            yes     3.0
12            yes     3.0
13            yes     3.0
14            yes     3.0
15            yes     3.0
16            yes     3.0
17            yes     3.0
18            yes     3.0
19            yes     3.0


In [ ]:
print("Количество непустых значений в каждом столбце:")
gdf.notnull().sum()

Количество непустых значений в каждом столбце:


,0
id,946
@id,946
house_number,480
district,155
street,367
building,946
floors,946
geometry,946


In [ ]:
gdf.to_file("baikalsk_floors.geojson", driver="GeoJSON")

In [ ]:
def fix_building_type(row):
    if row['building'] == 'yes':
        if row['floors'] >= 3:
            return 'apartments'
        elif row['floors'] == 2:
            return 'house'
        else:
            return 'house'
    return row['building']

gdf['building'] = gdf.apply(fix_building_type, axis=1)

In [ ]:
gdf.to_file("baikalsk_building.geojson", driver="GeoJSON")

### Новые столбцы

In [ ]:
gdf = gpd.read_file("baikalsk_building.geojson")
gdf.head()

,id,@id,house_number,district,street,building,floors,geometry
0,way/233398440,way/233398440,None,None,None,train_station,1.0,"POLYGON ((104.22613 51.50023, 104.22651 51.500..."
1,way/295659409,way/295659409,169,микрорайон Гагарина,None,apartments,3.0,"POLYGON ((104.15024 51.52243, 104.1501 51.5224..."
2,way/295659410,way/295659410,192,микрорайон Гагарина,None,apartments,3.0,"POLYGON ((104.15338 51.51682, 104.15244 51.516..."
3,way/295659411,way/295659411,197,микрорайон Гагарина,None,apartments,3.0,"POLYGON ((104.15638 51.51899, 104.1552 51.5189..."
4,way/295659413,way/295659413,203,микрорайон Гагарина,None,apartments,3.0,"POLYGON ((104.15733 51.51971, 104.15614 51.519..."


In [ ]:
gdf_proj = gdf.to_crs(epsg=32648)
centroids = gdf_proj.geometry.centroid
centroids = gpd.GeoSeries(centroids, crs=32648).to_crs(epsg=4326)

gdf['lat'] = centroids.y
gdf['lon'] = centroids.x

In [ ]:
def reverse_geocode(lat, lon):
    try:
        r = requests.get(
            "https://nominatim.openstreetmap.org/reverse",
            params={'lat': lat, 'lon': lon, 'format': 'json', 'addressdetails': 1},
            headers={'User-Agent': 'baikalsk-gis-project'}
        )
        addr = r.json().get('address', {})
        return {
            'street': addr.get('road'),
            'house_number': addr.get('house_number'),
            'district': addr.get('suburb') or addr.get('neighbourhood'),
        }
    except:
        return None

In [ ]:
for idx, row in gdf[gdf['street'].isna()].iterrows():
    if pd.notna(row['lat']) and pd.notna(row['lon']):
        result = reverse_geocode(row['lat'], row['lon'])
        if result:
            for field in ['street', 'house_number', 'district']:
                if pd.isna(row[field]):
                    gdf.at[idx, field] = result[field]
        time.sleep(1)

In [ ]:
print("Количество непустых значений в каждом столбце:")
gdf.notnull().sum()

Количество непустых значений в каждом столбце:


,0
id,946
@id,946
house_number,555
district,383
street,644
building,946
floors,946
geometry,946
lat,946
lon,946


In [ ]:
gdf_proj = gdf.to_crs(epsg=32648)
gdf['area_m2'] = gdf_proj.geometry.area
gdf['perimeter_m'] = gdf_proj.geometry.length

gdf.head()

,id,@id,house_number,district,street,building,floors,geometry,lat,lon,area_m2,perimeter_m
0,way/233398440,way/233398440,None,None,Привокзальная улица,train_station,1.0,"POLYGON ((104.22613 51.50023, 104.22651 51.500...",51.500247,104.226334,257.637720,72.267813
1,way/295659409,way/295659409,169,микрорайон Гагарина,None,apartments,3.0,"POLYGON ((104.15024 51.52243, 104.1501 51.5224...",51.522375,104.150359,1524.880690,219.932765
2,way/295659410,way/295659410,192,микрорайон Гагарина,None,apartments,3.0,"POLYGON ((104.15338 51.51682, 104.15244 51.516...",51.516761,104.152909,879.007163,158.067681
3,way/295659411,way/295659411,197,микрорайон Гагарина,None,apartments,3.0,"POLYGON ((104.15638 51.51899, 104.1552 51.5189...",51.518916,104.155791,1280.699267,195.643669
4,way/295659413,way/295659413,203,микрорайон Гагарина,None,apartments,3.0,"POLYGON ((104.15733 51.51971, 104.15614 51.519...",51.519639,104.156735,1279.656179,195.604930


In [ ]:
gdf.to_file("baikalsk_analiz_zastroyki.geojson", driver="GeoJSON")

### Районы

In [ ]:
!pip install osmnx geopandas shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 1.2 MB/s eta 0:00:00


In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import requests
import numpy as np
import json
import os
from shapely.geometry import box

In [ ]:
gdf = ox.features_from_place(
    "Байкальск, Иркутская область, Россия",
    tags={"place": ["neighbourhood", "suburb", "quarter"]}
)

gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
gdf = gdf.to_crs(epsg=4326)

if "name" in gdf.columns:
    name_col = "name"
elif "name:ru" in gdf.columns:
    name_col = "name:ru"
else:
    name_col = None

if name_col:
    gdf["district_name"] = gdf[name_col].fillna("Неизвестно")
else:
    gdf["district_name"] = [f"Район {i+1}" for i in range(len(gdf))]

gdf = gdf[["district_name", "geometry"]].reset_index(drop=True)

# ~ означает НЕ, то есть оставляем только те строки, где геометрия валидна И не пустая
gdf = gdf[gdf.geometry.is_valid & ~gdf.geometry.is_empty].copy()
gdf.to_file("baikalsk_neighborhoods.geojson", driver="GeoJSON")

print(f"Районов: {len(gdf)}")
print(gdf["district_name"].to_string())

Районов: 4
0     микрорайон Гагарина
1    микрорайон Строитель
2        микрорайон Южный
3            Красный Ключ


## Функциональное наполнение территории

In [ ]:
!pip install geopandas

### Отели

In [ ]:
import pandas as pd
import geopandas as gpd

columns = ['id', 'name', 'functional_type', 'address', 'lat', 'lon', 'rating', 'services', 'floors']
df = pd.DataFrame(columns=columns)

In [ ]:
def add_object(df, obj_id, name, functional_type, address, lat=None, lon=None, rating=None, services=None, floors=None):
    df = pd.concat([df, pd.DataFrame([{
        'id': obj_id,
        'name': name,
        'functional_type': functional_type,
        'address': address,
        'lat': lat,
        'lon': lon,
        'rating': rating,
        'services': services,
        'floors': floors
    }])], ignore_index=True)
    return df

In [ ]:
df = add_object(df, 1, "Белый соболь", "Park Hotel", "Микрорайон Красный ключ, 95, Байкальск, Слюдянский район, Иркутская область, 665932", 51.510765, 104.119583, 4.8, "​Завтрак, ​Можно с животными,​ Сауны, ​Бани, ​Детская комната, ​Прачечная​, Банкетный зал, ​Своя парковка, ​Сейф, ​Хранение багажа, Бассейн, Wi-Fi для клиентов, ​Круглосуточная стойка регистрации, ​Кафе", 2)
df = add_object(df, 2, "Baikal Best Edelweiss", "Hotel", "Микрорайон Красный ключ, 66, Байкальск, Слюдянский район, Иркутская область, 665932", 51.516926, 104.124554, 4.6, "​Завтрак, ​Своя парковка, ​Хранение багажа, ​Ski-room (хранение лыж и сноубордов), Wi-Fi для клиентов, ​Трансфер, ​Круглосуточная стойка регистрации, ​Кафе", 3)
df = add_object(df, 3, "Космос", "Hotel", "Микрорайон Гагарина, 2а, Байкальск, Слюдянский район, Иркутская область", 51.523078, 104.148662, 4.6, "​Завтрак, ​Можно с животными​, Детская комната, ​Прачечная, ​Пляж, ​Конференц-зал​, Своя парковка, ​Сейф, ​Хранение багажа, ​Wi-Fi для клиентов, ​Круглосуточная стойка регистрации, ​Кафе", 2)
df = add_object(df, 4, "Соболинка", "Park Hotel", "Микрорайон Красный ключ, 90, Байкальск, Слюдянский район, Иркутская область, 665932", 51.51500, 104.14020, 4.6, "​Завтрак, ​SPA, ​Можно с животными, ​Сауны, ​Детская комната, ​Прачечная, ​Кондиционер в номерах, Wi-Fi для клиентов​, Круглосуточная стойка регистрации, ​Туалет, ​Кафе", 4)
df = add_object(df, 5, "Ski-let", "The hotel complex", "Микрорайон Гагарина, 218, Байкальск, Слюдянский район, Иркутская область, 665932", 51.5238, 104.146, 4.8, "​Сауны, ​Бани​, Своя парковка, ​Хранение багажа, ​Финская парная, ​Бассейн, ​Беседки, ​Комната отдыха, ​TV/DVD​, Полотенца​, Простыни​, Тапочки​, Wi-Fi, ​Трансфер", 2)
df = add_object(df, 6, "Гранд Байкал", "Hotel", "​Гора Соболиная, ​Микрорайон Красный ключ, 91л, Байкальск, Слюдянский район, Иркутская область, 665932", 51.5091, 104.12, 4.4, "​Завтрак, ​Можно с животными, ​Детская комната, ​Прачечная, ​Кондиционер в номерах, Wi-Fi для клиентов, ​Круглосуточная стойка регистрации, ​Туалет, ​Кафе", 2)
df = add_object(df, 7, "Уютный двор", "Hotel", "Р-258 147 километр, 3, Байкальск, Слюдянский район, Иркутская область, 665932", 51.515356, 104.138304, 3.1, "​Номер на час, ​Без звёзд, ​Завтрак, ​Можно с животными, ​Прачечная​, Банкетный зал​, Своя парковка, Wi-Fi для клиентов, ​Круглосуточная стойка регистрации, ​Кафе", 2)
df = add_object(df, 8, "Куршавель", "Hotel", "1-й квартал, 6, Южный м-н, Байкальск, Слюдянский район, Иркутская область, 665930", 51.5057, 104.145, 4.8, "​Можно с животными, ​Своя парковка, ​Wi-Fi для клиентов, ​Круглосуточная стойка регистрации", 2)
df = add_object(df, 9, "Маргобай", "Hotel", "​Березовый переулок, 9, Строитель м-н, Байкальск, Слюдянский район, Иркутская область, 665932", 51.5244, 104.137, 4.7, "Завтрак, ​Коливинг, ​Можно с животными​, Сауны, ​Прачечная, ​Банкетный зал, ​Конференц-зал, ​Своя парковка, Бассейн, Wi-Fi для клиентов, ​Круглосуточная стойка регистрации, ​Туалет, ​Кафе", 2)
df = add_object(df, 10, "Мельница", "Hotel", "​Микрорайон Красный ключ, 9, Байкальск, Слюдянский район, Иркутская область, 665932", 51.515, 104.123, 4.6, "​SPA, ​Сауны, ​Бани​, Прачечная, ​Своя парковка, ​Гараж, Бассейн, ​Wi-Fi для клиентов, ​Круглосуточная стойка регистрации, Кафе", 3)
df = add_object(df, 11, "Уют", "Hotel", "Строительная, 13, Строитель м-н, Байкальск, Слюдянский район, Иркутская область, 665932", 51.521610, 104.132465, 4.1, "​Коливинг, ​Можно с животными, ​Сауны, ​Прачечная​, Своя парковка, Wi-Fi для клиентов, ​Кафе", 2)
df = add_object(df, 12, "Орлинка", "Hotel", "1-й квартал, 27а, Южный м-н, Байкальск, Слюдянский район, Иркутская область, 665930", 51.5150, 104.1402, 4.8, "Можно с животными, ​Детская комната​, Прачечная, ​Своя парковка, ​Гараж, ​Wi-Fi для клиентов, ​Круглосуточная стойка регистрации, ​Туалет​, Кафе", 2)
df = add_object(df, 13, "Хамар Дабан", "Hotel", "Р-258 144 километр, 1, Байкальск, Слюдянский район, Иркутская область, 665932", 51.523398, 104.098554, 4.4, "Можно с животными, ​Детская комната, ​Прачечная​, Конференц-зал, ​Своя парковка, ​Детская площадка, ​Wi-Fi для клиентов,​ Круглосуточная стойка регистрации, ​Кафе", 3)
df = add_object(df, 14, "Северные склоны", "Hotel", "​Речная, 11, Байкальск, Слюдянский район, Иркутская область, 665932", 51.521633, 104.138682, 4.8, "​Аренда номера, ​Завтрак, ​Можно с животными, ​Сауны, ​Беседки​, Стиральная машина, ​Своя парковка, ​Wi-Fi для клиентов​, Трансфер​, Кафе", 3)
df = add_object(df, 15, "Берёза", "Hotel", "Березовый переулок, 14/1, Байкальск, Слюдянский район, Иркутская область, 665932", 51.524283, 104.138062, 4.8, "​Аренда номера, ​Коттедж, ​Бани, ​Беседки​, Стиральная машина, ​Своя парковка, Wi-Fi для клиентов", 2)
df = add_object(df, 16, "Дом на Байкале", "Hotel", "Байкальская улица, 13а, Байкальск, Слюдянский район, Иркутская область, 665932", 51.522978, 104.136310, 5, "Аренда номера, ​Коттедж, ​Беседки, ​Стиральная машина, ​Кондиционер в номерах, ​Своя парковка, ​Хранение багажа, ​Детская площадка, Wi-Fi для клиентов, ​Трансфер", 1)
df = add_object(df, 17, "Волна", "Hotel", "Байкальская улица, 14, Байкальск, Слюдянский район, Иркутская область, 665932", 51.523897, 104.137469, 4.8, "​Аренда номера, ​Коттедж, ​Завтрак, ​Можно с животными, ​Беседки, ​Стиральная машина, ​Своя парковка, ​Хранение багажа, Wi-Fi для клиентов, ​Трансфер, ​Кафе", 2)
df = add_object(df, 18, "Алёнкин дом", "Hotel", "Байкальская улица, 6, Байкальск, Слюдянский район, Иркутская область, 665932", 51.523622, 104.139804, 4.8, "Аренда номера, ​Коттедж, ​Бани, ​Беседки, ​Стиральная машина, ​Кондиционер в номерах, ​Своя парковка​, Хранение багажа, Wi-Fi для клиентов, ​Трансфер", 2)
df = add_object(df, 19, "Шишки", "Hotel", "Ленская улица, 17, пос. Утулик, Слюдянский район, Иркутская область, 665932", 51.524283, 104.138062, 4.8, "​Аренда номера, ​Коттедж, ​Завтрак​, Бани, ​Беседки, ​Своя парковка​, Банный чан​, Детская площадка, Бассейн, Прокат велосипедов, Wi-Fi для клиентов​, Кафе​", 3)
df = add_object(df, 20, "Байкальские рассветы", "Hotel", "Набережная, 28, Байкальск, Слюдянский район, Иркутская область, 665932", 51.525813, 104.136570, 5, "​Аренда номера, ​Коттедж, ​Беседки, ​Стиральная машина, ​Своя парковка", 2)
df = add_object(df, 21, "Байкальский дом", "Hotel", "Микрорайон Красный ключ, 22, Байкальск, Слюдянский район, Иркутская область, 665932", 51.515854, 104.123105, 5, "​Аренда номера, ​Коттедж, ​Можно с животными, ​Бани, ​Беседки​, Стиральная машина, ​Своя парковка, ​Хранение багажа, Wi-Fi для клиентов", 1)
df = add_object(df, 22, "Нефрит", "Hotel", "7-я Байкальская улица, 10, Байкальск, Слюдянский район, Иркутская область, 665932", 51.525062, 104.120751, 4.9, "​Аренда номера​, Коттедж, ​Можно с животными, ​Сауны, ​Беседки, ​На берегу​, Своя парковка, ​Хранение багажа, ​Детская площадка, ​Wi-Fi для клиентов, ​Трансфер", 2)
df = add_object(df, 23, "Кедр", "Hotel", "​Микрорайон Красный ключ, 69Б, Байкальск, Слюдянский район, Иркутская область, 665932", 51.514677, 104.122449, 5, "​Аренда номера, ​Беседки, ​Своя парковка, Wi-Fi для клиентов", 3)
df = add_object(df, 24, "Домашний очаг", "Hotel", "Байкальская улица, 25в, Байкальск, Слюдянский район, Иркутская область, 665932", 51.523443, 104.132771, 4.8, "Аренда номера, ​Коттедж, ​Завтрак, ​Можно с животными, ​Бани, ​Беседки, ​Стиральная машина, ​Своя парковка, ​Хранение багажа, Wi-Fi для клиентов, ​Трансфер​, Кафе", 2)
df = add_object(df, 25, "Синяя крыша", "Hotel", "Микрорайон Красный ключ, 87, Байкальск, Слюдянский район, Иркутская область, 665932", 51.518080, 104.124569, 4.5, "Аренда номера, ​Стиральная машина, Wi-Fi для клиентов", 2)
df = add_object(df, 26, "Хижина Дяди Сережи", "Hotel", "Микрорайон Красный ключ, 47, Байкальск, Слюдянский район, Иркутская область, 665932", 51.517558, 104.121703, 5, "​Аренда номера, ​Коттедж, ​Можно с животными, ​Бани, ​Беседки​, Стиральная машина, ​Своя парковка, Wi-Fi для клиентов", 2)
df = add_object(df, 27, "Green house", "Hotel", "​Микрорайон Красный ключ, 48, Байкальск, Слюдянский район, Иркутская область, 665932", 51.515126, 104.122099, 5, "​Аренда номера, ​Коттедж, ​Стиральная машина, ​Своя парковка, ​Хранение багажа, Wi-Fi для клиентов", 3)
df = add_object(df, 28, "Апарт-бюро", "Hotel", "Микрорайон Красный ключ, 10а, Байкальск, Слюдянский район, Иркутская область, 665932", 51.514784, 104.123491, 5, "​Аренда номера, ​Беседки, ​Стиральная машина​, Своя парковка, Wi-Fi для клиентов", 2)
df = add_object(df, 29, "Колыбель Байкала", "Hotel", "7-я Байкальская улица, 6, Байкальск, Слюдянский район, Иркутская область, 665932", 51.524978, 104.118227, 5, "​Аренда номера, ​Коттедж, ​Можно с животными, ​Бани, ​Беседки​, Своя парковка, ​Хранение багажа, ​Детская площадка", 2)
df = add_object(df, 30, "Домик у озера", "Hotel", "Российская улица, 2, пос. Утулик, Слюдянский район, Иркутская область, 665932", 51.552349, 104.060321, 4.9, "Аренда номера, ​Бани, ​Стиральная машина, ​Своя парковка, ​Сейф, ​Хранение багажа​, Детская площадка, Wi-Fi для клиентов", 2)
df = add_object(df, 31, "Байкал Йети", "Hotel", "​Береговая улица, 12, пос. Утулик, Слюдянский район, Иркутская область, 665932", 51.518774, 104.144305, 4.9, "​Аренда номера, ​Коттедж​, Завтрак, ​Можно с животными, ​Бани, ​Своя парковка, Wi-Fi для клиентов", 2)
df = add_object(df, 32, "Гостевой дом", "Hotel", "Микрорайон Красный ключ, 14, Байкальск, Слюдянский район, Иркутская область, 665932", 51.516858, 104.120895, None, "Аренда номера, ​Коттедж​, Сауны, ​Стиральная машина, ​Своя парковка", 2)

/tmp/ipykernel_5933/2495579128.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([{


In [ ]:
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['lon'], df['lat']), crs="EPSG:4326")

df.to_csv("hotels_baikalsk.csv", index=False)
gdf.to_file("hotels_baikalsk.geojson", driver="GeoJSON")

gdf.head(5)

    id                   name    functional_type  \
0    1           Белый соболь         Park Hotel   
1    2  Baikal Best Edelweiss              Hotel   
2    3                 Космос              Hotel   
3    4              Соболинка         Park Hotel   
4    5                Ski-let  The hotel complex   
5    6           Гранд Байкал              Hotel   
6    7            Уютный двор              Hotel   
7    8              Куршавель              Hotel   
8    9               Маргобай              Hotel   
9   10               Мельница              Hotel   
10  11                    Уют              Hotel   
11  12                Орлинка              Hotel   
12  13            Хамар Дабан              Hotel   
13  14        Северные склоны              Hotel   
14  15                 Берёза              Hotel   
15  16         Дом на Байкале              Hotel   
16  17                  Волна              Hotel   
17  18            Алёнкин дом              Hotel   
18  19      

### Кафе

In [ ]:
!pip install bs4
!pip install requests

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import base64
import json
import time
import geopandas as gpd
from geopy.geocoders import Nominatim

In [ ]:
headers = {"User-Agent": "Mozilla/5.0"}
geolocator = Nominatim(user_agent="baikalsk_map")

In [ ]:
def parse_page(url):
    soup = BeautifulSoup(requests.get(url, headers=headers).text, "html.parser")
    data = []

    for item in soup.find_all("li", class_="company-list__i"):
        name_tag = item.find("span", class_="company-info-name-org")
        name = name_tag.text.strip() if name_tag else ""
        if not name:
            continue

        address_tag = item.find("address", class_="company-info-address-full")
        address = address_tag.text.strip() if address_tag else ""

        time_tag = item.find("span", class_="company-info-time-full")
        working_hours = time_tag.get("data-text-after", "").strip() if time_tag else ""

        phone = ""
        phone_tag = item.find("a", class_="company-info-phone")
        if phone_tag and phone_tag.get("data-props"):
            phone = ", ".join(json.loads(phone_tag.get("data-props")).get("phones", []))

        website = ""
        site_tag = item.find("a", class_="company-info-site-open")
        if site_tag and site_tag.get("data-link"):
            website = base64.b64decode(site_tag.get("data-link")).decode("utf-8")

        services = [s.text.strip() for s in item.find_all("li", class_="company-info-category")]

        data.append({
            "name": name,
            "category": "cafe",
            "address": address,
            "working_hours": working_hours,
            "phone": phone,
            "website": website,
            "services": ", ".join(services)
        })

    return pd.DataFrame(data)

In [ ]:
df = pd.concat([
    parse_page("https://bajkalsk.jsprav.ru/kafe/"),
    parse_page("https://bajkalsk.jsprav.ru/kafe/page-2/")
], ignore_index=True).drop_duplicates(subset=["name", "address"]).reset_index(drop=True)

In [ ]:
def clean_address(addr):
    if not addr:
        return None
    for old, new in [("микрорайон", ""), ("мкр.", ""), ("посёлок", ""),
                     ("поселок", ""), ("Россия,", "")]:
        addr = addr.replace(old, new)
    return addr.strip()

In [ ]:
def geocode_multi(name, address):
    queries = [
        f"{address}, Байкальск, Иркутская область, Россия",
        f"{clean_address(address)}, Байкальск",
        f"{name}, Байкальск",
        f"{name}, Иркутская область",
    ]
    for q in queries:
        try:
            loc = geolocator.geocode(q, timeout=10)
            time.sleep(1)
            if loc:
                return loc.latitude, loc.longitude
        except:
            continue
    return None, None

In [ ]:
lats, lons = [], []
for i, row in df.iterrows():
    lat, lon = geocode_multi(row["name"], row["address"])
    lats.append(lat)
    lons.append(lon)

In [ ]:
df["lat"] = lats
df["lon"] = lons

In [ ]:
df_good = df.dropna(subset=["lat", "lon"]).reset_index(drop=True)
df_bad = df[df["lat"].isna()].reset_index(drop=True)

print(f"Найдено: {len(df_good)} / {len(df)}")
print(f"Не найдено: {len(df_bad)}")

df_good["id"] = ["cafe_" + str(i) for i in df_good.index]

Найдено: 28 / 32
Не найдено: 4


In [ ]:
gdf = gpd.GeoDataFrame(
    df_good,
    geometry=gpd.points_from_xy(df_good["lon"], df_good["lat"]),
    crs="EPSG:4326"
)

df_good.to_csv("cafe_baikalsk_clean.csv", index=False)
df_bad.to_csv("cafe_baikalsk_not_found.csv", index=False)
gdf.to_file("cafe_baikalsk.geojson", driver="GeoJSON")

### Парки


In [ ]:
import pandas as pd
import geopandas as gpd

columns = ['id', 'name', 'functional_type', 'address', 'lat', 'lon', 'rating', 'description']
df = pd.DataFrame(columns=columns)

def add_object(df, obj_id, name, functional_type, address, lat=None, lon=None, rating=None, description=None):
    df = pd.concat([df, pd.DataFrame([{
        'id': obj_id,
        'name': name,
        'functional_type': functional_type,
        'address': address,
        'lat': lat,
        'lon': lon,
        'rating': rating,
        'description': description
    }])], ignore_index=True)
    return df

df = add_object(df, 1, "Парк Искусств", "park", "Гагарина м-н, Байкальск, Слюдянский район, Иркутская область, 665932", 51.520237, 104.153621, 4.9, "Парк Искусств — это культурно-просветительское пространство, созданное для того, чтобы объединить природу, творчество и вдохновение. Это место, где искусство становится частью повседневной жизни, а прогулка превращается в эстетическое и эмоциональное переживание.")
df = add_object(df, 2, "Парк Целлюлозников", "park", "Байкальск, Слюдянский район, Иркутская область, 665932", 51.520456, 104.147260, 4.9, "Парк Целлюлозников — это общественный парк, созданный в честь работников целлюлозно-бумажной промышленности, которая сыграла важную роль в развитии местной экономики и жизни города. Парк представляет собой зелёную зону для отдыха, прогулок и культурных мероприятий, оформленную с уважением к трудовой истории региона.")
df = add_object(df, 3, "Парк Укрощение строптивой реки", "park", "Байкальск, Слюдянский район, Иркутская область", 51.51978, 104.141263, 5, None)
df = add_object(df, 4, "Сквер", "park", "Байкальск, Слюдянский район, Иркутская область", 51.520609, 104.139792, 4.9, None)
df = add_object(df, 5, "Татьянин сквер", "park", "Байкальск, Слюдянский район, Иркутская область", 51.508819, 104.141474, 4.9, "Сквер назван в честь техника-озеленителя Татьяны Сергеевны Миротворской (1914-1994 гг.).")
df = add_object(df, 6, "Сквер Первостроителей", "park", "Байкальск, Слюдянский район, Иркутская область", 51.521425, 104.135503, 3.9, "От первых палаток, от просеки первой, От рук мастерства и от жара сердец, Родился у моря, у гор, в тайге вечной Байкальск-город юности, город-боец! Нэлли Тихонова. Основан 19 мая 2017 г.")
df = add_object(df, 7, "Парк воинской славы", "park", "Южный м-н, Байкальск, Слюдянский район, Иркутская область, 665930", 51.511583, 104.144574, None, "Парк воинской славы — это мемориальный парк, созданный в честь памяти защитников Отечества, павших в различных войнах, прежде всего в Великой Отечественной. Это место, где сочетаются природа, история и глубокое уважение к подвигу солдат, отдавших жизнь за Родину.")

gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['lon'], df['lat']), crs="EPSG:4326")

df.to_csv("parks_baikalsk.csv", index=False)
gdf.to_file("parks_baikalsk.geojson", driver="GeoJSON")

print(gdf.head())

  id                            name functional_type  \
0  1                   Парк Искусств            park   
1  2              Парк Целлюлозников            park   
2  3  Парк Укрощение строптивой реки            park   
3  4                           Сквер            park   
4  5                  Татьянин сквер            park   

                                             address        lat         lon  \
0  Гагарина м-н, Байкальск, Слюдянский район, Ирк...  51.520237  104.153621   
1  Байкальск, Слюдянский район, Иркутская область...  51.520456  104.147260   
2     Байкальск, Слюдянский район, Иркутская область  51.519780  104.141263   
3     Байкальск, Слюдянский район, Иркутская область  51.520609  104.139792   
4     Байкальск, Слюдянский район, Иркутская область  51.508819  104.141474   

   rating                                        description  \
0     4.9  Парк Искусств — это культурно-просветительское...   
1     4.9  Парк Целлюлозников — это общественный парк, со...

/tmp/ipykernel_5933/3700402873.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([{


### Музеи. Достопримечательности: Природные объекты, музеи, памятники, культурные центры (расположение, описание, посещаемость).

In [ ]:
import pandas as pd
import geopandas as gpd

columns = ['id', 'name', 'functional_type', 'address', 'rating', 'lat', 'lon']
df = pd.DataFrame(columns=columns)

In [ ]:
def add_object(df, obj_id, name, functional_type, address, rating=None, lat=None, lon=None):
    df = pd.concat([df, pd.DataFrame([{
        'id': obj_id,
        'name': name,
        'functional_type': functional_type,
        'address': address,
        'rating': rating,
        'lat': lat,
        'lon': lon
    }])], ignore_index=True)
    return df

In [ ]:
df = add_object(df, 1, 'Байкальский народный театр', 'Многофункциональный культурный центр', 'Железнодорожная улица, 4а, Байкальск, Слюдянский район, Иркутская область, 665932', 4.6, 51.518943, 104.140074)
df = add_object(df, 2, 'Смотровая площадка', 'Смотровая площадка', 'Байкальск, Слюдянский район, Иркутская область', 4.5, 51.52408, 104.162262)
df = add_object(df, 3, 'Смотровая площадка Обзорка Закатная', 'Смотровые площадки', 'Байкальск, Слюдянский район, Иркутская область', 4.9, 51.527631, 104.170427)
df = add_object(df, 4, 'Мурал Дом клубники', 'Арт-объект', 'Парк Целлюлозников, микрорайон Гагарина, 28, Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', 5, 51.520692, 104.146614)
df = add_object(df, 5, 'Арт-объект Глубина', 'Арт-объект', 'Байкальск, Слюдянский район, Иркутская область', 4.8, 51.523711, 104.182076)
df = add_object(df, 6, 'Арт-объект Эко.цех', 'Арт-объект', '​Байкальск, Слюдянский район, Иркутская область', 4, 51.51718, 104.189016)
df = add_object(df, 7, 'Байкальский целлюлозный завод', 'Промышленное предприятие', '​Байкальск, Слюдянский район, Иркутская область', 5, 51.519781, 104.178262)
df = add_object(df, 8, 'Мурал У озера', 'Арт-объект', 'Микрорайон Гагарина, 202, Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', 5, 51.5189, 104.156714)
df = add_object(df, 9, 'Храм Троицы Живоначальной', 'Церкви', '3-й квартал, 23, Байкальск, Слюдянский район, Иркутская область', None, 51.510905, 104.145590)
df = add_object(df, 10, 'Граффити Пейзаж Байкальской природы', 'Арт-объект', 'Байкальск, Слюдянский район, Иркутская область', None, 51.521014, 104.156797)
df = add_object(df, 11, 'Граффити Тигр', 'Арт-объект', 'Байкальск, Слюдянский район, Иркутская область', None, 51.516969, 104.155189)
df = add_object(df, 12, 'Граффити Че Гевара', 'Арт-объект', 'Байкальск, Слюдянский район, Иркутская область', None, 51.516896, 104.155196)
df = add_object(df, 13, 'Декоративное сооружение Кот', 'Памятники и скульптуры', 'Байкальск, Слюдянский район, Иркутская область', 5, 51.51591, 104.155422)
df = add_object(df, 14, 'Декоративный объект Счастливые часов не наблюдают', 'Памятники и скульптуры', 'Южный м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.508341, 104.141491)
df = add_object(df, 15, 'Памятник Воинам, погибшим в боях за честь и свободу нашей родины', 'Памятники и скульптуры', 'Татьянин сквер, Южный м-н, Байкальск, Слюдянский район, Иркутская область', 5, 51.50935, 104.141424)
df = add_object(df, 16, 'Памятная доска Котову И.М.', 'Памятные доски', 'Микрорайон Гагарина, 184, Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.517758, 104.151782)
df = add_object(df, 17, 'Декоративный объект Старцы', 'Памятники и скульптуры', 'Южный м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.510308, 104.140474)
df = add_object(df, 18, 'Памятный камень Знание - сила', 'Памятники и скульптуры', 'Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', 5, 51.519411, 104.152409)
df = add_object(df, 19, 'Декоративный объект Туфля', 'Памятники и скульптуры', 'Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.519567, 104.14926)
df = add_object(df, 20, 'Декоративный объект Нотный стан', 'Памятники и скульптуры', 'Парк Искусств, Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', 5, 51.520478, 104.153097)
df = add_object(df, 21, 'Памятный камень в строительство стадиона им. Великой Победы', 'Памятники и скульптуры', 'Строитель м-н, Байкальск, Слюдянский район, Иркутская область', 3, 51.520316, 104.140936)
df = add_object(df, 22, 'Мемориальная доска Александру Николаевичу Бабученко', 'Памятные доски', 'Байкальск, Слюдянский район, Иркутская область', None, 51.519974, 104.144597)
df = add_object(df, 23, 'Декоративное сооружение Дерево Любви', 'Памятники и скульптуры', 'Байкальск, Слюдянский район, Иркутская область', 5, 51.519875, 104.146604)
df = add_object(df, 24, 'Памятная доска С.А. Антончику', 'Памятные доски', '4-й квартал, 1, Южный м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.509669, 104.1417)
df = add_object(df, 25, 'Памятная доска Л.Е Вокину', 'Памятные доски', '4-й квартал, 1, Южный м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.509691, 104.141714)
df = add_object(df, 26, 'Памятная доска В. В. Шишкину', 'Памятные доски', 'Микрорайон Гагарина, 10, Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.522457, 104.152012)
df = add_object(df, 27, 'Декоративный объект Орёл', 'Памятники и скульптуры', 'Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', 5, 51.522625, 104.149746)
df = add_object(df, 28, 'Декоративное сооружение Байкальск', 'Памятники и скульптуры', '​Строитель м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.521714, 104.140166)
df = add_object(df, 29, 'Декоративное сооружение Гном', 'Памятники и скульптуры', '​Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.522042, 104.156095)
df = add_object(df, 30, 'Памятная доска сквера Целлюлозников', 'Памятные доски', 'Парк Целлюлозников, Байкальск, Слюдянский район, Иркутская область', None, 51.520181, 104.147689)
df = add_object(df, 31, 'Декоративный объект Дедушка гриб', 'Памятники и скульптуры', 'Строитель м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.523437, 104.135662)
df = add_object(df, 32, 'Декоративный объект Зайцы', 'Памятники и скульптуры', 'Строитель м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.523313, 104.135757)
df = add_object(df, 33, 'Памятная доска Курбану Абдуловичу Иткулову', 'Памятные доски', 'Микрорайон Гагарина, 158, Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.520981, 104.154687)
df = add_object(df, 34, 'Декоративное сооружение Жираф', 'Памятники и скульптуры', 'Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', 4.9, 51.518581, 104.149999)
df = add_object(df, 35, 'Декоративный объект Слоник', 'Памятники и скульптуры', 'Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', 4.9, 51.518587, 104.150299)
df = add_object(df, 36, 'Скульптура Скрипач', 'Памятники и скульптуры', 'Парк Искусств, Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.520335, 104.153273)
df = add_object(df, 37, 'Декоративное сооружение Девочка на мяче', 'Памятники и скульптуры', 'Южный м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.508398, 104.143703)
df = add_object(df, 38, 'Декоративное сооружение Девочка с медведем', 'Памятники и скульптуры', 'Южный м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.508508, 104.14339)
df = add_object(df, 39, 'Декоративный объект Айболит', 'Памятники и скульптуры', 'Южный м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.508608, 104.142816)
df = add_object(df, 40, 'Декоративный объект Гриб', 'Памятники и скульптуры', 'Южный м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.510497, 104.140724)
df = add_object(df, 41, 'Декоративное сооружение Богатырь', 'Памятники и скульптуры', 'Байкальск, Слюдянский район, Иркутская область', None, 51.514083, 104.12264)
df = add_object(df, 42, 'Памятная доска Юрию Николаевичу Рыбину', 'Памятные доски', 'Микрорайон Гагарина, 211, Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.524143, 104.148178)
df = add_object(df, 43, 'Памятная доска Киногруппе фильма У озера', 'Памятные доски', '2-й квартал, 51, Южный м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.508393, 104.141272)
df = add_object(df, 44, 'Памятная доска В.Т. Москальчуку', 'Памятные доски', 'Микрорайон Гагарина, 172, Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.519622, 104.151183)
df = add_object(df, 45, 'Граффити Петербург', 'Арт-объект', 'Микрорайон Гагарина, 220, Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', 5, 51.523891, 104.149947)
df = add_object(df, 46, 'Памятная доска Строительству БЦБК', 'Памятные доски', 'Парк Целлюлозников, Байкальск, Слюдянский район, Иркутская область', None, 51.520047, 104.147279)
df = add_object(df, 47, 'Аллея молодожёнов', 'Природные достопримечательности', 'Парк Целлюлозников, Байкальск, Слюдянский район, Иркутская область', None, 51.520157, 104.14696)
df = add_object(df, 48, 'Декоративный объект Вселенная Байкала', 'Памятники и скульптуры', 'Байкальск, Слюдянский район, Иркутская область', 5, 51.528622, 104.147968)
df = add_object(df, 49, 'Арт-объект Тропы', 'Арт-объект', 'Байкальск, Слюдянский район, Иркутская область', 4.5, 51.523719, 104.161837)
df = add_object(df, 50, 'Арт-объект Место силы', 'Арт-объект', 'Байкальск, Слюдянский район, Иркутская область', None, 51.523427, 104.160583)
df = add_object(df, 51, 'Арт-объект Город', 'Арт-объект', 'Площадь Фестивальная, Байкальск, Слюдянский район, Иркутская область', 4.9, 51.523717, 104.148581)
df = add_object(df, 52, 'Арт-объект Свое дело', 'Арт-объект', 'Южный м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.508298, 104.141508)
df = add_object(df, 53, 'Арт-объект Чудесная земля', 'Арт-объект', 'Гора Соболиная, микрорайон Красный ключ, 91л/1, Байкальск, Слюдянский район, Иркутская область', None, 51.509327, 104.120533)
df = add_object(df, 54, 'Декоративный объект Синтез', 'Памятники и скульптуры', 'Татьянин сквер, Южный м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.508614, 104.141345)
df = add_object(df, 55, 'Декоративный объект Бременские музыканты', 'Памятники и скульптуры', 'Гагарина м-н, Байкальск, Слюдянский район, Иркутская область', 5, 51.521104, 104.151993)
df = add_object(df, 56, 'Топиарий Сибирский медведь', 'Памятники и скульптуры', 'Строитель м-н, Байкальск, Слюдянский район, Иркутская область', 4.9, 51.520721, 104.139836)
df = add_object(df, 57, 'Декоративное сооружение Верстовой Екатерининский столб', 'Памятники и скульптуры', 'Байкальск, Слюдянский район, Иркутская область', 5, 51.528215, 104.147432)
df = add_object(df, 58, 'Мемориальная доска Татьянин сквер', 'Памятные доски', 'Татьянин сквер, Южный м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.508549, 104.141275)
df = add_object(df, 59, 'Декоративный объект Цветы', 'Памятники и скульптуры', 'Южный м-н, Байкальск, Слюдянский район, Иркутская область', 5, 51.510289, 104.14095)
df = add_object(df, 60, 'Декоративный объект Весы', 'Памятники и скульптуры', 'Байкальск, Слюдянский район, Иркутская область', None, 51.519539, 104.178546)
df = add_object(df, 61, 'Мемориальный комплекс Связь поколений страны', 'Памятники и скульптуры', 'Южный м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.5113, 104.145011)
df = add_object(df, 62, 'Поклонный крест', 'Религиозные объекты', 'Южный м-н, Байкальск, Слюдянский район, Иркутская область', None, 51.511137, 104.146004)
df = add_object(df, 63, 'Юбилейный', 'Дом культуры', '​2-й квартал, 51, Байкальск, Слюдянский район, Иркутская область', 4.9, 51.508170, 104.141035)
df = add_object(df, 64, 'Музей природы южного Прибайкалья', 'Музей', 'Микрорайон Гагарина, 211, Байкальск, Слюдянский район, Иркутская область', 4.5, 51.524244, 104.147674)
df = add_object(df, 65, 'Гора Соболиная', 'Смотровые площадки', 'Гора Соболиная, Слюдянский район, Иркутская область', 5, 51.492933, 104.11028)
df = add_object(df, 66, 'Историко-краеведческий музей им. В. Т. Москальчука', 'Музей', '2-й квартал, 51, Байкальск, Слюдянский район, Иркутская область', None, 51.508170, 104.141035)
df.head()

/tmp/ipykernel_5933/1016056463.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([{


,id,name,functional_type,address,rating,lat,lon
0,1,Байкальский народный театр,Многофункциональный культурный центр,"Железнодорожная улица, 4а, Байкальск, Слюдянск...",4.6,51.518943,104.140074
1,2,Смотровая площадка,Смотровая площадка,"Байкальск, Слюдянский район, Иркутская область",4.5,51.524080,104.162262
2,3,Смотровая площадка Обзорка Закатная,Смотровые площадки,"Байкальск, Слюдянский район, Иркутская область",4.9,51.527631,104.170427
3,4,Мурал Дом клубники,Арт-объект,"Парк Целлюлозников, микрорайон Гагарина, 28, Г...",5.0,51.520692,104.146614
4,5,Арт-объект Глубина,Арт-объект,"Байкальск, Слюдянский район, Иркутская область",4.8,51.523711,104.182076


In [ ]:
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['lon'], df['lat']), crs="EPSG:4326")

df.to_csv("places_baikalsk.csv", index=False)
gdf.to_file("places_baikalsk.geojson", driver="GeoJSON")

### Медицинские учреждения

In [ ]:
import pandas as pd
import geopandas as gpd

columns = ['id', 'name', 'functional_type', 'address', 'info', 'lat', 'lon']
df = pd.DataFrame(columns=columns)

In [ ]:
def add_object(df, obj_id, name, functional_type, address, info=None, lat=None, lon=None):
    df = pd.concat([df, pd.DataFrame([{
        'id': obj_id,
        'name': name,
        'functional_type': functional_type,
        'address': address,
        'info': info,
        'lat': lat,
        'lon': lon
    }])], ignore_index=True)
    return df

In [ ]:
df = add_object(df, 1, 'Слюдянская больница', 'Больницы', '​Микрорайон Красный ключ, 92, Байкальск, Слюдянский район, Иркутская область', 'ОГБУЗ Слюдянская районная больница', 51.512071, 104.121946)
df = add_object(df, 2, 'Слюдянская больница', '​Детская консультация', 'Микрорайон Красный ключ, 92, Байкальск, Слюдянский район, Иркутская область', 'ОГБУЗ Слюдянская районная больница', 51.512071, 104.121946)
df = add_object(df, 3, 'Детская поликлиника', 'Поликлиника', 'Микрорайон Красный ключ, 92, Байкальск, Слюдянский район, Иркутская область', 'ОГБУЗ Слюдянская районная больница', 51.512071, 104.121946)
df = add_object(df, 4, 'Зубная фея', 'Стоматологическая клиника', 'Микрорайон Гагарина, 1, Байкальск, Слюдянский район, Иркутская область', 'Нет данных', 51.522714, 104.151114)
df = add_object(df, 5, 'Стоматологическая клиника', 'Частные стоматологии', 'Микрорайон Гагарина, 170, Байкальск, Слюдянский район, Иркутская область', 'Нет данных', 51.521868, 104.150045)
df = add_object(df, 6, 'Кабинет УЗИ', '​Медицинская диагностика', 'Микрорайон Гагарина, 214, Байкальск, Слюдянский район, Иркутская область', 'Нет данных', 51.522535, 104.147269)

/tmp/ipykernel_5933/225763996.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([{


In [ ]:
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['lon'], df['lat']), crs="EPSG:4326")

df.to_csv("hospitals_baikalsk.csv", index=False)
gdf.to_file("hospitals_baikalsk.geojson", driver="GeoJSON")

## Транспортная доступность

### дороги и границы

In [ ]:
!pip install osmnx geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.1 MB/s eta 0:00:00


In [ ]:
import osmnx as ox
import geopandas as gpd

place = "Baikalsk, Irkutsk Oblast, Russia"

city = ox.geocode_to_gdf(place)
boundary = city[["geometry"]]

boundary.to_file("baikalsk_boundary.geojson", driver="GeoJSON")

In [ ]:
import osmnx as ox
import geopandas as gpd

center = city.geometry.centroid.iloc[0]

# Скачивание дорог в радиусе 10 км"
graph = ox.graph_from_point((center.y, center.x), dist=10000, network_type="all", simplify=True)

edges = ox.graph_to_gdfs(graph, nodes=False, edges=True)
edges_gdf = edges[["geometry", "highway"]]

edges_gdf.to_file("baikal_roads.geojson", driver="GeoJSON")

/tmp/ipykernel_5933/3418429846.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = city.geometry.centroid.iloc[0]


In [ ]:
import geopandas as gpd

roads = gpd.read_file("baikal_roads.geojson")
boundary = gpd.read_file("baikalsk_boundary.geojson")

roads = roads.to_crs(boundary.crs)
clipped = gpd.clip(roads, boundary)

clipped.to_file("baikal_roads_clipped.geojson", driver="GeoJSON")

### Маршруты и общественный транспорт

In [ ]:
!pip install bs4
!pip install requests

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
from geopy.geocoders import Nominatim
import time

In [ ]:
BASE_URL = "https://baykalsk.ginfo.ru"
START_URL = "https://baykalsk.ginfo.ru/marshruty/"
headers = {"User-Agent": "Mozilla/5.0"}
geolocator = Nominatim(user_agent="baikalsk_stops")
SKIP_ROUTES = ("102", "103", "544")

In [ ]:
def get_routes():
    response = requests.get(START_URL, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    routes = []

    for link in soup.select("a.mrt_link"):
        route_name = link.text.strip()

        if route_name.startswith(("102", "103", "544")):
            continue

        route_url = urljoin(BASE_URL, link.get("href"))

        routes.append({
            "route": route_name,
            "url": route_url
        })

    return routes

In [ ]:
def parse_route(route):
    response = requests.get(route["url"], headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    data = []

    directions = soup.select("div.dir_stations")

    for dir_idx, direction in enumerate(directions, start=1):
        stops = direction.select("a.station_link")

        for order, stop in enumerate(stops, start=1):
            stop_name = stop.text.strip()
            stop_url = urljoin(BASE_URL, stop.get("href"))

            data.append({
                "route": route["route"],
                "direction": dir_idx,
                "stop_order": order,
                "stop_name": stop_name.replace(f"{order}.", "").strip(),
                "stop_url": stop_url
            })

    return data

In [ ]:
routes = get_routes()
print(f"Найдено маршрутов: {len(routes)}")

all_data = []

for r in routes:
    print(f"Парсим маршрут {r['route']}")
    route_data = parse_route(r)
    all_data.extend(route_data)

df = pd.DataFrame(all_data)
df.to_csv("baikalsk_routes_stops.csv", index=False)

Найдено маршрутов: 3
Парсим маршрут 5
Парсим маршрут 6
Парсим маршрут 4


In [ ]:
import pandas as pd
df = pd.read_csv("baikalsk_routes_stops.csv")
print(df)

    route  direction  stop_order      stop_name  \
0       5          1           1       Гагарина   
1       5          1           2  Спорткомплекс   
2       5          1           3      Общежития   
3       5          1           4            МЖК   
4       5          1           5    Автостанция   
..    ...        ...         ...            ...   
68      4          2           9         Ракета   
69      4          2          10         202 км   
70      4          2          11           Дачи   
71      4          2          12         Солзан   
72      4          2          13     Ж/д вокзал   

                                             stop_url  
0     https://baykalsk.ginfo.ru/ostanovki/2/gagarina/  
1   https://baykalsk.ginfo.ru/ostanovki/9/sportkom...  
2   https://baykalsk.ginfo.ru/ostanovki/20/obschez...  
3         https://baykalsk.ginfo.ru/ostanovki/4/mzhk/  
4   https://baykalsk.ginfo.ru/ostanovki/19/avtosta...  
..                                                .

Остановки и их маршруты

In [ ]:
BASE_URL = "https://baykalsk.ginfo.ru"
STOPS_URL = "https://baykalsk.ginfo.ru/ostanovki/"
headers = {"User-Agent": "Mozilla/5.0"}

In [ ]:
def get_stops():
    response = requests.get(STOPS_URL, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    stops = []

    for link in soup.select("a.ost_link"):
        stop_name = link.text.strip()
        stop_url = urljoin(BASE_URL, link.get("href"))

        stops.append({
            "stop_name": stop_name,
            "stop_url": stop_url
        })
    return stops

In [ ]:
def parse_stop_routes(stop):
    response = requests.get(stop["stop_url"], headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    stop_routes = []

    # автобусы
    for link in soup.select("div.mrt_avtobus a.mrt_link"):
        route_name = link.text.strip()
        if route_name.startswith(("102", "103", "544")):
            continue
        stop_routes.append({
            "stop_name": stop["stop_name"],
            "route": route_name
        })

    # маршрутки
    for link in soup.select("div.mrt_marshrutka a.mrt_link"):
        route_name = link.text.strip()
        if route_name.startswith(("102", "103", "544")):
            continue
        stop_routes.append({
            "stop_name": stop["stop_name"],
            "route": route_name
        })

    return stop_routes

In [ ]:
stops = get_stops()
print(f"Найдено остановок: {len(stops)}")

all_data = []

for stop in stops:
  routes = parse_stop_routes(stop)
  all_data.extend(routes)

df_stops_routes = pd.DataFrame(all_data)
df_stops_routes.to_csv("baikalsk_stops_routes.csv", index=False)

Найдено остановок: 20


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

BASE_URL = "https://baykalsk.ginfo.ru"
STOPS_URL = "https://baykalsk.ginfo.ru/ostanovki/"

headers = {"User-Agent": "Mozilla/5.0"}

def get_stops():
    response = requests.get(STOPS_URL, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    stops = []

    for link in soup.select("a.ost_link"):
        stop_name = link.text.strip()
        stop_url = urljoin(BASE_URL, link.get("href"))

        stops.append({
            "stop_name": stop_name,
            "stop_url": stop_url
        })
    return stops

def parse_stop_routes(stop):
    response = requests.get(stop["stop_url"], headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    stop_routes = []

    # автобусы
    for link in soup.select("div.mrt_avtobus a.mrt_link"):
        route_name = link.text.strip()
        if route_name.startswith(("102", "103", "544")):
            continue
        stop_routes.append({
            "stop_name": stop["stop_name"],
            "route": route_name
        })

    # маршрутки
    for link in soup.select("div.mrt_marshrutka a.mrt_link"):
        route_name = link.text.strip()
        if route_name.startswith(("102", "103", "544")):
            continue
        stop_routes.append({
            "stop_name": stop["stop_name"],
            "route": route_name
        })

    return stop_routes

stops = get_stops()
print(f"Найдено остановок: {len(stops)}")

all_data = []

for stop in stops:
    try:
        routes = parse_stop_routes(stop)
        all_data.extend(routes)
    except Exception as e:
        print(f"Ошибка на остановке {stop['stop_name']}: {e}")

df_stops_routes = pd.DataFrame(all_data)

df_stops_routes.to_csv("baikalsk_stops_routes.csv", index=False)

Найдено остановок: 20


KeyboardInterrupt: 

In [ ]:
df = pd.read_csv("baikalsk_stops_routes.csv")
unique_stops = df["stop_name"].unique()
coords_dict = {}

In [ ]:
for stop in unique_stops:
    try:
        loc = geolocator.geocode(f"{stop}, Baikalsk, Irkutsk Oblast, Russia")
        coords_dict[stop] = (loc.latitude, loc.longitude) if loc else (None, None)
    except:
        coords_dict[stop] = (None, None)
    time.sleep(1)

In [ ]:
df["lat"] = df["stop_name"].map(lambda x: coords_dict[x][0])
df["lon"] = df["stop_name"].map(lambda x: coords_dict[x][1])
df.to_csv("baikalsk_stops_routes_coords.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_csv("baikalsk_stops_routes_coords.csv")

missing_lat = df["lat"].isna().sum()
missing_lon = df["lon"].isna().sum()

print(f"Пустых lat: {missing_lat}")
print(f"Пустых lon: {missing_lon}")

Пустых lat: 11
Пустых lon: 11


In [ ]:
import pandas as pd

df = pd.read_csv("baikalsk_stops_routes_coords.csv")

missing_rows = df[df["lat"].isna() | df["lon"].isna()]

print(missing_rows)

            stop_name  route  lat  lon
3             Бабха-2      6  NaN  NaN
7              Горный      6  NaN  NaN
8                Дачи      4  NaN  NaN
9          Ж/д вокзал      4  NaN  NaN
19  Площадь Юбилейная      5  NaN  NaN
20  Площадь Юбилейная      6  NaN  NaN
21  Площадь Юбилейная      4  NaN  NaN
22         Пост ГИБДД      5  NaN  NaN
23         Пост ГИБДД      6  NaN  NaN
28         Сангородок      5  NaN  NaN
29         Сангородок      6  NaN  NaN


In [ ]:
coords_manual = {
    "Бабха-2": (None, None),
    "Горный": (None, None),
    "Дачи": (51.498685, 104.210784),
    "Ж/д вокзал": (51.500202, 104.227218),
    "Площадь Юбилейная": (51.50934, 104.141624),
    "Пост ГИБДД": (51.515728, 104.13762),
    "Сангородок": (51.51294, 104.122205),
}

In [ ]:
for stop, (lat, lon) in coords_manual.items():
    if lat is not None and lon is not None:
        df.loc[df["stop_name"] == stop, "lat"] = lat
        df.loc[df["stop_name"] == stop, "lon"] = lon

In [ ]:
to_remove = ["Бабха-2", "Горный"]

df = df[~df["stop_name"].isin(to_remove)]

In [ ]:
df.to_csv("baikalsk_stops_routes_coords_final.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_csv("baikalsk_stops_routes_coords_final.csv")
print(df)

            stop_name  route        lat         lon
0         Автостанция      5  51.520465  104.133230
1         Автостанция      6  51.520465  104.133230
2         Автостанция      4  51.520465  104.133230
3            Гагарина      5  51.519289  104.150424
4            Гагарина      6  51.519289  104.150424
5            Гагарина      4  51.519289  104.150424
6                Дачи      4  51.498685  104.210784
7          Ж/д вокзал      4  51.500202  104.227218
8                Маяк      5  51.519580  104.138856
9                Маяк      6  51.519580  104.138856
10               Маяк      4  51.519580  104.138856
11                МЖК      5  51.523757  104.134368
12                МЖК      6  51.523757  104.134368
13                МЖК      4  51.523757  104.134368
14          Общежития      5  51.523250  104.139792
15          Общежития      6  51.523250  104.139792
16          Общежития      4  51.523250  104.139792
17  Площадь Юбилейная      5  51.509340  104.141624
18  Площадь 